**Problema de Negócio:** O painel de produtividade do armazém está mostrando que os operadores estão batendo recordes de separação (Picking), mas as docas de expedição estão vazias. O diagnóstico aponta que coletores de dados (scanners) estão com defeito, registrando múltiplos bipes (cliques duplos) na mesma caixa em questão de milissegundos.

**Objetivo:** Usar PySpark para aplicar uma janela de desduplicação (remover os "falsos bipes") e usar SQL para calcular o tempo real de turno e o UPH (Units Per Hour - Caixas separadas por hora) de cada operador, revelando quem são os verdadeiros talentos e quem precisa de retreinamento.

In [0]:
%python
import pandas as pd
import random
from datetime import datetime, timedelta

# Fixando semente para dados controlados
random.seed(42)

operadores = ['OP-001 (Maria)', 'OP-002 (João)', 'OP-003 (Carlos)']
data_bruta = []
inicio_turno = datetime(2026, 9, 17, 8, 0, 0) # Turno começando às 8h da manhã

for op in operadores:
    if 'Maria' in op:
        qtd_caixas, chance_duplo_clique = 130, 0.45  
    elif 'João' in op:
        qtd_caixas, chance_duplo_clique = 60, 0.02   
    else:
        qtd_caixas, chance_duplo_clique = 210, 0.01  
        
    tempo_atual = inicio_turno
    
    for i in range(qtd_caixas):
        box_id = f"BOX-BR-{random.randint(100000, 999999)}"
        
        tempo_atual += timedelta(seconds=random.randint(30, 120))
        
        data_bruta.append({'worker_id': op, 'box_id': box_id, 'scan_timestamp': tempo_atual})
        
        if random.random() < chance_duplo_clique:
            tempo_duplicado = tempo_atual + timedelta(seconds=1)
            data_bruta.append({'worker_id': op, 'box_id': box_id, 'scan_timestamp': tempo_duplicado})

df_bronze = spark.createDataFrame(pd.DataFrame(data_bruta))
df_bronze.createOrReplaceTempView("bronze_picking")

display(df_bronze)

worker_id,box_id,scan_timestamp
OP-001 (Maria),BOX-BR-770487,2026-09-17T08:00:44.000Z
OP-001 (Maria),BOX-BR-770487,2026-09-17T08:00:45.000Z
OP-001 (Maria),BOX-BR-388389,2026-09-17T08:01:45.000Z
OP-001 (Maria),BOX-BR-388389,2026-09-17T08:01:46.000Z
OP-001 (Maria),BOX-BR-872246,2026-09-17T08:02:28.000Z
OP-001 (Maria),BOX-BR-671858,2026-09-17T08:03:09.000Z
OP-001 (Maria),BOX-BR-133326,2026-09-17T08:03:42.000Z
OP-001 (Maria),BOX-BR-133326,2026-09-17T08:03:43.000Z
OP-001 (Maria),BOX-BR-343962,2026-09-17T08:05:16.000Z
OP-001 (Maria),BOX-BR-688508,2026-09-17T08:06:11.000Z


In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze = spark.table("bronze_picking")

w = Window.partitionBy("box_id").orderBy("scan_timestamp")

df_silver = df_bronze.withColumn("num_bipe", F.row_number().over(w)) \
                     .filter(F.col("num_bipe") == 1) \
                     .drop("num_bipe")

df_silver.createOrReplaceTempView("silver_picking")

bipes_brutos = df_bronze.count()
caixas_reais = df_silver.count()

print(f"🚨 Leituras brutas no sistema WMS (com erros): {bipes_brutos}")
print(f"✅ Caixas reais separadas fisicamente (após limpeza): {caixas_reais}")
print(f"🗑️ Falsos bipes removidos: {bipes_brutos - caixas_reais}")

display(df_silver)

🚨 Leituras brutas no sistema WMS (com erros): 464
✅ Caixas reais separadas fisicamente (após limpeza): 400
🗑️ Falsos bipes removidos: 64


worker_id,box_id,scan_timestamp
OP-002 (João),BOX-BR-100425,2026-09-17T08:51:39.000Z
OP-001 (Maria),BOX-BR-102260,2026-09-17T09:54:19.000Z
OP-003 (Carlos),BOX-BR-102689,2026-09-17T11:25:30.000Z
OP-002 (João),BOX-BR-105814,2026-09-17T09:07:35.000Z
OP-003 (Carlos),BOX-BR-106182,2026-09-17T08:26:16.000Z
OP-001 (Maria),BOX-BR-106814,2026-09-17T08:10:25.000Z
OP-003 (Carlos),BOX-BR-108491,2026-09-17T12:31:13.000Z
OP-002 (João),BOX-BR-110139,2026-09-17T08:00:44.000Z
OP-003 (Carlos),BOX-BR-112861,2026-09-17T09:58:33.000Z
OP-003 (Carlos),BOX-BR-118080,2026-09-17T09:34:50.000Z


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW gold_produtividade AS
WITH Calculo_Turno AS (
    SELECT 
        worker_id,
        COUNT(box_id) AS total_caixas_separadas,
        (unix_timestamp(MAX(scan_timestamp)) - unix_timestamp(MIN(scan_timestamp))) / 3600.0 AS horas_trabalhadas
    FROM silver_picking
    GROUP BY worker_id
)

SELECT 
    worker_id, 
    total_caixas_separadas, 
    ROUND(horas_trabalhadas, 2) AS horas_trabalhadas,
    ROUND(total_caixas_separadas / horas_trabalhadas, 1) AS caixas_por_hora
FROM Calculo_Turno;

SELECT * FROM gold_produtividade ORDER BY caixas_por_hora DESC;

worker_id,total_caixas_separadas,horas_trabalhadas,caixas_por_hora
OP-002 (João),60,1.20,50.1
OP-001 (Maria),130,2.64,49.3
OP-003 (Carlos),210,4.57,45.9


In [0]:
%sql
SELECT 
    worker_id,
    caixas_por_hora,
    CASE 
        WHEN caixas_por_hora >= 45.0 THEN '🚀 Alta Performance (>45)'
        WHEN caixas_por_hora >= 35.0 THEN '✅ Dentro da Meta (35-44)'
        ELSE '⚠️ Abaixo da Meta (Retreinamento)'
    END AS avaliacao_desempenho
FROM gold_produtividade
ORDER BY caixas_por_hora DESC;

worker_id,caixas_por_hora,avaliacao_desempenho
OP-002 (João),50.1,🚀 Alta Performance (>45)
OP-001 (Maria),49.3,🚀 Alta Performance (>45)
OP-003 (Carlos),45.9,🚀 Alta Performance (>45)


Databricks visualization. Run in Databricks to view.